In [17]:
# 1. Import thư viện

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [2]:
# 2. Import file Excel

from pathlib import Path

file_path = Path(r"C:\Users\Admin\Downloads\cookie_cats.xlsx")

if not file_path.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

# Xem tên các sheet trong workbook
xls = pd.ExcelFile(file_path)
print("Các sheet:", xls.sheet_names)

Các sheet: ['cookie_cats']


In [3]:
# 3. Đọc dữ liệu

sheet_name = xls.sheet_names[0]
df = pd.read_excel(file_path, sheet_name=sheet_name)

print("Đã đọc sheet:", sheet_name)
print("Kích thước dữ liệu:", df.shape)
display(df.head())

Đã đọc sheet: cookie_cats
Kích thước dữ liệu: (90189, 1)


,"userid,version,sum_gamerounds,retention_1,retention_7"
0,"116,gate_30,3,False,False"
1,"337,gate_30,38,True,False"
2,"377,gate_40,165,True,False"
3,"483,gate_40,1,False,False"
4,"488,gate_40,179,True,True"


In [4]:
# 4. Đọc lại dữ liệu và tách cột

sheet_name = xls.sheet_names[0]

# Đọc toàn bộ nội dung dưới dạng text
raw = pd.read_excel(
    file_path,
    sheet_name=sheet_name,
    header=None,
    dtype=str
)

# Tách dòng CSV đang bị dồn trong một cột thành các cột riêng
df = raw.iloc[:, 0].str.split(",", expand=True)

# Lấy dòng đầu tiên làm tên cột
df.columns = df.iloc[0].str.strip()
df = df.iloc[1:].reset_index(drop=True)

# Bỏ hàng tiêu đề thừa nếu có
df = df[df["userid"] != "userid"].reset_index(drop=True)

print("Kích thước dữ liệu:", df.shape)
display(df.head())

Kích thước dữ liệu: (90189, 5)


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [5]:
# 4. Kiểm tra tên cột

print("Tên cột:")
print(df.columns.tolist())

Tên cột:
['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


In [6]:
# 5. Chuyển kiểu dữ liệu

df["userid"] = pd.to_numeric(df["userid"], errors="coerce")
df["sum_gamerounds"] = pd.to_numeric(
    df["sum_gamerounds"],
    errors="coerce"
)

for column in ["retention_1", "retention_7"]:
    df[column] = (
        df[column]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )

In [7]:
# 6. Kiểm tra cấu trúc và dữ liệu thiếu

print("Thông tin dữ liệu:")
df.info()

print("\nGiá trị thiếu:")
print(df.isnull().sum())

print("\nSố dòng trùng hoàn toàn:", df.duplicated().sum())
print("Số user ID bị lặp:", df["userid"].duplicated().sum())

Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB

Giá trị thiếu:
0
userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64

Số dòng trùng hoàn toàn: 0
Số user ID bị lặp: 0


In [18]:
# 7. Kiểm tra số người chơi trong mỗi nhóm

print("Số người chơi trong mỗi nhóm:")
print(df["version"].value_counts())

Số người chơi trong mỗi nhóm:
version
gate_40    45489
gate_30    44699
Name: count, dtype: int64


In [19]:
# 8. Thống kế game rounds theo từng nhóm

print("Game rounds summary by group:")
print(df.groupby("version")["sum_gamerounds"].describe())

Game rounds summary by group:
           count       mean         std  min  25%   50%   75%     max
version                                                              
gate_30  44699.0  51.342111  102.057598  0.0  5.0  17.0  50.0  2961.0
gate_40  45489.0  51.298776  103.294416  0.0  5.0  16.0  52.0  2640.0


In [10]:
# 9. Loại bỏ outlier

print("Top 5 game rounds:", df['sum_gamerounds'].nlargest(5).tolist())

df = df[df['sum_gamerounds'] < df['sum_gamerounds'].max()]

print("\nPlayers with 0 rounds:", (df['sum_gamerounds'] == 0).sum())

summary = df.groupby('version').agg(
    players=('userid', 'count'),
    d1_retention=('retention_1', 'mean'),
    d7_retention=('retention_7', 'mean'),
    mean_rounds=('sum_gamerounds', 'mean'),
    median_rounds=('sum_gamerounds', 'median')
)
print(summary)

Top 5 game rounds: [49854, 2961, 2640, 2438, 2294]

Players with 0 rounds: 3994
         players  d1_retention  d7_retention  mean_rounds  median_rounds
version                                                                 
gate_30    44699      0.448198      0.190183    51.342111           17.0
gate_40    45489      0.442283      0.182000    51.298776           16.0


In [13]:
from scipy import stats

In [14]:
# 10. Retention tests

for metric in ["retention_7", "retention_1"]:
    table = pd.crosstab(df["version"], df[metric])
    chi2, p_value, dof, expected = stats.chi2_contingency(table)

    print(f"{metric}:")
    print(table)
    print(f"Chi-square p-value: {p_value:.4f}\n")

retention_7:
retention_7  False  True 
version                  
gate_30      36198   8501
gate_40      37210   8279
Chi-square p-value: 0.0016

retention_1:
retention_1  False  True 
version                  
gate_30      24665  20034
gate_40      25370  20119
Chi-square p-value: 0.0750



In [15]:
# 11. Game rounds test

g30 = df.loc[df["version"] == "gate_30", "sum_gamerounds"]
g40 = df.loc[df["version"] == "gate_40", "sum_gamerounds"]

statistic, p_value = stats.mannwhitneyu(
    g30,
    g40,
    alternative="two-sided"
)

print(f"Mann–Whitney U statistic: {statistic:.0f}")
print(f"Game rounds p-value: {p_value:.4f}")

Mann–Whitney U statistic: 1024285762
Game rounds p-value: 0.0509


In [16]:
# 12. Tóm tắt kết quả

results = []

for metric in ["retention_7", "retention_1"]:
    table = pd.crosstab(df["version"], df[metric])
    chi2, p_value, dof, expected = stats.chi2_contingency(table)

    results.append({
        "metric": metric,
        "test": "Chi-square",
        "p_value": p_value
    })

results.append({
    "metric": "sum_gamerounds",
    "test": "Mann–Whitney U",
    "p_value": stats.mannwhitneyu(
        g30,
        g40,
        alternative="two-sided"
    ).pvalue
})

results = pd.DataFrame(results)
display(results)

,metric,test,p_value
0,retention_7,Chi-square,0.001639
1,retention_1,Chi-square,0.075010
2,sum_gamerounds,Mann–Whitney U,0.050892
